# PlateVision AI Pro — CNN **V3.1** | Google Colab

**Objectif :** traiter les confusions observées sur V3 (notamment `J` → `U/I/4/1`) en diversifiant les familles de polices à l'entraînement et en pondérant légèrement les caractères difficiles. **Aucun gain n'est garanti.**

**Référence V3 observée :** 91,86 % d'accuracy sur **3 600 caractères synthétiques Lato** ; rappel de `J` : 8 % (8/100). Ce n'est **pas** une mesure sur des plaques photographiées. L'ancien V2 (80,65 %) utilisait un autre benchmark : ne pas comparer les scores directement.

### Démarrage rapide
1. Ouvrir dans Google Colab ; choisir un GPU (T4 si disponible).
2. **Exécution → Tout exécuter.** L'installation des polices peut prendre quelques minutes ; le CNN V3.1 s'entraîne automatiquement.
3. Télécharger l'archive V3.1 générée. **Le test final DejaVu reste désactivé.**
4. Pour une comparaison calculée dans la même session, ajouter **l'archive V3 précédente** dans `/content`, puis réexécuter la cellule de comparaison (ou activer son option d'import avant « Tout exécuter »).

**Protocole gelé :** validation = groupe `Lato` (neuf fichiers d'origine) ; test final = groupe `DejaVu` (quatre fichiers d'origine), jamais généré ni consulté au cours de l'entraînement. Train = `Liberation` + nouvelles familles distinctes. Validation reproduite avec **le même générateur et la même graine** que V3 ; toute différence de versions Pillow/polices/runtime peut toutefois affecter sa reproduction exacte.

**Protection :** aucun fichier de V1/V2/V3 n'est modifié. Nouveau répertoire d'exécution et nouveau checkpoint V3.1.

## 1 — Configuration

Les graines et le contrat d'entrée sont fixés. Pour cette expérience, les deux interventions principales sont l'élargissement des polices d'entraînement et une **faible pondération** de caractères difficiles. Ce protocole ne permet pas d'isoler causalement l'effet de chacune.

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
from io import BytesIO
import hashlib, json, os, random, string, sys, tempfile, uuid, zipfile, subprocess, warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from sklearn.metrics import classification_report, confusion_matrix
import sklearn
import tensorflow as tf

SEED = 2026
CLASSES = string.digits + string.ascii_uppercase
NUM_CLASSES = 36
IMAGE_SIZE = 32
N_TRAIN_PER_CLASS = 600      # 21 600 images d'entraînement, avant pondération
N_VALID_PER_CLASS = 100      # 3 600 images ; exactement les paramètres V3
N_TEST_PER_CLASS = 100       # générées UNIQUEMENT après décision finale
BATCH_SIZE = 64
MAX_EPOCHS = 30
RUN_TRAIN = True
DOWNLOAD_EXPORT = True

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print('Déterminisme complet non garanti :', type(exc).__name__)

BASE_OUTPUT = Path('/content/platevision_v31_runs') if Path('/content').exists() else Path.cwd() / 'platevision_v31_runs'
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = BASE_OUTPUT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT = RUN_DIR / 'platevision_cnn_v31.keras'

print('Python :', sys.version.split()[0], '| TensorFlow :', tf.__version__, '| scikit-learn :', sklearn.__version__)
print('GPU :', tf.config.list_physical_devices('GPU'))
print('Dossier V3.1 :', RUN_DIR)
assert CLASSES == '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'

## 2 — Installer les polices supplémentaires

V3 avait seulement trois groupes : `Liberation` (train), `Lato` (validation), `DejaVu` (test). Pour améliorer la diversité sans contaminer la validation ou le test, on tente d'installer notamment **Roboto**, **Carlito** et **FreeSans** en plus des trois familles existantes. La cellule imprime les échecs éventuels et un contrôle strict suivra.

In [ ]:
RUN_FONT_INSTALL = True
PACKAGES = [
    'fonts-dejavu-core', 'fonts-liberation', 'fonts-lato',
    'fonts-roboto', 'fonts-crosextra-carlito', 'fonts-freefont-ttf',
    'fonts-croscore',
]

if RUN_FONT_INSTALL:
    if not Path('/usr/bin/apt-get').exists():
        print('apt-get absent : vérifier manuellement les polices Linux déjà installées.')
    else:
        result = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, text=True)
        if result.returncode:
            warnings.warn('apt-get update a échoué : ' + (result.stderr or result.stdout)[-400:])
        for package in PACKAGES:
            result = subprocess.run(
                ['apt-get', 'install', '-y', '-qq', package], capture_output=True, text=True
            )
            print(('OK    ' if result.returncode == 0 else 'ÉCHEC ') + package)
            if result.returncode:
                print((result.stderr or result.stdout)[-200:])
        subprocess.run(['fc-cache', '-f'], capture_output=True, text=True)
print('Installation terminée. Inventaire des polices dans la cellule suivante.')

## 3 — Reconstituer les groupes V3 et geler validation/test

Les **mêmes neuf fichiers Lato** et les **quatre fichiers DejaVu** que dans le manifeste de ton export V3 sont sélectionnés. Ils ne seront **jamais** utilisés pour produire le train V3.1. Une nouvelle installation ne doit pas changer les fichiers retenus pour les évaluations.

In [ ]:
ALLOWED_FAMILIES = {
    'DejaVu Sans', 'DejaVu Sans Condensed', 'DejaVu Sans Mono',
    'Liberation Sans', 'Liberation Sans Narrow', 'Liberation Mono',
    'Lato', 'Roboto', 'Roboto Condensed', 'Open Sans',
    'Open Sans Condensed', 'Arimo', 'Carlito', 'Clear Sans',
    'Clear Sans Light', 'Clear Sans Medium', 'Noto Sans',
    'Noto Sans Display', 'Noto Sans Mono', 'PT Sans',
    'PT Sans Narrow', 'FreeSans', 'Cousine',
}
STEM_PREFIXES = (
    'dejavu', 'liberation', 'lato', 'roboto', 'opensans', 'arimo',
    'carlito', 'clearsans', 'notosans-', 'notosansdisplay',
    'notosansmono', 'pts', 'ptn', 'freesans', 'cousine',
)

def family_group(family):
    name = family.casefold()
    for prefix, key in (
        ('dejavu', 'DejaVu'), ('liberation', 'Liberation'),
        ('noto', 'Noto'), ('roboto', 'Roboto'),
        ('open sans', 'Open Sans'), ('pt sans', 'PT Sans'),
        ('clear sans', 'Clear Sans'),
    ):
        if name.startswith(prefix):
            return key
    return family

FONT_ROOT = Path('/usr/share/fonts/truetype')
if not FONT_ROOT.is_dir():
    raise RuntimeError('Ce notebook demande un environnement Linux Google Colab avec /usr/share/fonts/truetype.')
font_groups = defaultdict(list)
font_metadata = {}
for path in sorted(FONT_ROOT.rglob('*.ttf')):
    if any(word in path.name.lower() for word in ('italic', 'oblique', 'script', 'symbol')):
        continue
    if not path.stem.casefold().startswith(STEM_PREFIXES):
        continue
    try:
        family, style = ImageFont.truetype(str(path), 30).getname()
        if family not in ALLOWED_FAMILIES:
            continue
        font = ImageFont.truetype(str(path), 30)
        if not all(font.getbbox(char) is not None for char in CLASSES):
            continue
    except (OSError, ValueError):
        continue
    key = family_group(family)
    font_groups[key].append(path)
    font_metadata[str(path)] = {'family': family, 'style': style, 'group': key}

# Gel explicite des mêmes noms de fichiers que dans manifest_v3.json.
ORIGINAL_LATO_FILES = {
    'Lato-Black.ttf', 'Lato-Bold.ttf', 'Lato-Hairline.ttf',
    'Lato-Heavy.ttf', 'Lato-Light.ttf', 'Lato-Medium.ttf',
    'Lato-Regular.ttf', 'Lato-Semibold.ttf', 'Lato-Thin.ttf',
}
ORIGINAL_DEJAVU_FILES = {
    'DejaVuSans-Bold.ttf', 'DejaVuSans.ttf',
    'DejaVuSansMono-Bold.ttf', 'DejaVuSansMono.ttf',
}
for key, required in [('Lato', ORIGINAL_LATO_FILES), ('DejaVu', ORIGINAL_DEJAVU_FILES)]:
    matched = sorted(p for p in font_groups[key] if p.name in required)
    names = [p.name for p in matched]
    if len(names) != len(required) or set(names) != required:
        raise RuntimeError(f'Fichiers {key} V3 manquants/différents : {sorted(required - set(names))}. Corriger avant toute comparaison.')
    font_groups[key] = matched

# Écarter les grands paquets multi-scriptes (notamment Noto), dont certaines fontes
# pourraient afficher un glyphe manquant malgré un getbbox() non nul.
CURATED_TRAIN_GROUPS = {'Liberation', 'Roboto', 'Carlito', 'FreeSans', 'Arimo',
                        'Cousine', 'Clear Sans', 'Open Sans', 'PT Sans'}
train_groups = sorted(set(font_groups) & CURATED_TRAIN_GROUPS)
if 'Liberation' not in train_groups or len(train_groups) < 3:
    raise RuntimeError(
        f'Entraînement insuffisamment diversifié : groupes={train_groups}. '
        'Vérifier l’installation de Roboto, Carlito et FreeSans ci-dessus.'
    )
SPLIT_GROUPS = {
    'train': train_groups,
    'validation': ['Lato'],
    'test': ['DejaVu'],
}
assert not (set(train_groups) & {'Lato', 'DejaVu'})
assert set(SPLIT_GROUPS['validation']).isdisjoint(SPLIT_GROUPS['test'])
for split, groups in SPLIT_GROUPS.items():
    print(f'{split:11}: {groups} | {sum(len(font_groups[g]) for g in groups)} fichier(s)')
print('Fichiers exacts de validation :', [p.name for p in font_groups['Lato']])
print('Jeu de test : réservé et NON généré.')

# Empreintes des polices réellement utilisées, pour audit et reproductibilité.
font_hashes = {
    str(p): hashlib.sha256(p.read_bytes()).hexdigest()
    for paths in font_groups.values() for p in paths
}

## 4 — Générer les données et contrôler l'absence de fuite

La **validation Lato** est générée à l'aide du **générateur V3 inchangé**, avec le même nombre d'exemples par classe (`100`) et la même graine (`SEED + 202`). Cela permet une comparaison intra-session V3/V3.1 si tu fournis le checkpoint V3.

L'entraînement V3.1 sélectionne d'abord un **groupe** puis un **fichier de police** : les nombreuses variantes d'une seule famille ne dominent pas toutes les autres. On ajoute seulement aux images du **train** quelques altérations de largeur de trait et d'inclinaison. Le test final DejaVu reste absent de la mémoire jusqu'à la décision finale.

In [ ]:
# Fonction identique à celle de PlateVision_AI_CNN_V3_Ready.ipynb.
# Ne pas modifier : elle définit les données de validation et du test final V3.
def make_character_image_v3(char: str, font_path: Path, rng: np.random.Generator) -> np.ndarray:
    font = ImageFont.truetype(str(font_path), size=int(rng.integers(26, 38)))
    canvas = Image.new('L', (48, 48), color=255)
    draw = ImageDraw.Draw(canvas)
    left, top, right, bottom = draw.textbbox((0, 0), char, font=font)
    width, height = right - left, bottom - top
    dx, dy = rng.integers(-3, 4, size=2)
    origin = ((48 - width) / 2 - left + int(dx), (48 - height) / 2 - top + int(dy))
    gray_ink = int(rng.integers(0, 51))
    draw.text(origin, char, font=font, fill=gray_ink)
    canvas = canvas.rotate(float(rng.uniform(-9, 9)), resample=Image.Resampling.BICUBIC, fillcolor=255)
    canvas = canvas.resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC)
    if rng.random() < 0.30:
        canvas = canvas.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.15, 0.65))))
    if rng.random() < 0.55:
        canvas = ImageEnhance.Contrast(canvas).enhance(float(rng.uniform(0.78, 1.22)))
    if rng.random() < 0.40:
        canvas = ImageEnhance.Brightness(canvas).enhance(float(rng.uniform(0.88, 1.12)))
    array = np.asarray(canvas, dtype=np.float32)
    if rng.random() < 0.50:
        array += rng.normal(0, rng.uniform(1.0, 7.0), size=array.shape)
    if rng.random() < 0.12:
        temp = Image.fromarray(np.clip(array, 0, 255).astype(np.uint8), mode='L')
        buf = BytesIO()
        temp.save(buf, format='JPEG', quality=int(rng.integers(55, 91)))
        buf.seek(0)
        with Image.open(buf) as decoded:
            array = np.asarray(decoded.convert('L'), dtype=np.float32)
    return np.clip(array, 0, 255).astype(np.uint8)


def make_character_image_v31_train(char: str, font_path: Path, rng: np.random.Generator) -> np.ndarray:
    original = make_character_image_v3(char, font_path, rng)
    # Modifications TRAIN uniquement. Les valeurs restent de type uint8, caractère foncé sur blanc.
    img = Image.fromarray(original)
    r = rng.random()
    if r < 0.075:
        img = img.filter(ImageFilter.MinFilter(3))  # traits un peu plus épais
    elif r < 0.15:
        img = img.filter(ImageFilter.MaxFilter(3))  # traits un peu plus fins
    if rng.random() < 0.18:
        # Cisaillement faible : simulation d'une segmentation légèrement inclinée.
        shear = float(rng.uniform(-0.10, 0.10))
        img = img.transform((IMAGE_SIZE, IMAGE_SIZE), Image.Transform.AFFINE,
                            (1, shear, -shear * IMAGE_SIZE / 2, 0, 1, 0),
                            resample=Image.Resampling.BICUBIC, fillcolor=255)
    return np.asarray(img, dtype=np.uint8)


def generate_split(split: str, n_per_class: int, seed: int, training: bool = False):
    local_rng = np.random.default_rng(seed)
    files = [p for group in SPLIT_GROUPS[split] for p in font_groups[group]]
    count = n_per_class * NUM_CLASSES
    raw = np.empty((count, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    labels = np.empty(count, dtype=np.int32)
    offset = 0
    for label, char in enumerate(CLASSES):
        for _ in range(n_per_class):
            if training:
                group = train_groups[int(local_rng.integers(0, len(train_groups)))]
                group_files = font_groups[group]
                font_path = group_files[int(local_rng.integers(0, len(group_files)))]
                raw[offset] = make_character_image_v31_train(char, font_path, local_rng)
            else:
                # Exactement le tirage de V3 sur ses groupes de validation/test.
                font_path = files[int(local_rng.integers(0, len(files)))]
                raw[offset] = make_character_image_v3(char, font_path, local_rng)
            labels[offset] = label
            offset += 1
    order = local_rng.permutation(count)
    raw, labels = raw[order], labels[order]
    inputs = (1.0 - raw.astype(np.float32) / 255.0)[..., np.newaxis]
    assert inputs.shape == (count, 32, 32, 1)
    assert np.isfinite(inputs).all() and 0 <= inputs.min() <= inputs.max() <= 1
    assert np.array_equal(np.bincount(labels, minlength=36), np.full(36, n_per_class))
    print(f'{split:11}: {count:,} images | {n_per_class} par classe | {inputs.nbytes/2**20:.1f} Mio')
    return inputs, labels, raw

X_train, y_train, raw_train = generate_split('train', N_TRAIN_PER_CLASS, SEED + 1111, training=True)
X_val, y_val, raw_val = generate_split('validation', N_VALID_PER_CLASS, SEED + 202, training=False)

h_train = {hashlib.sha256(a.tobytes()).digest() for a in raw_train}
h_val = {hashlib.sha256(a.tobytes()).digest() for a in raw_val}
overlap = len(h_train & h_val)
print('Doublons exacts train/validation :', overlap)
if overlap:
    raise RuntimeError(f'Fuite potentielle : {overlap} image(s) identique(s) entre train et validation.')

validation_digest = hashlib.sha256(raw_val.tobytes() + y_val.tobytes()).hexdigest()
manifest = {
    'experiment': 'PlateVision CNN V3.1 — caractères synthétiques isolés',
    'seed': SEED,
    'image_shape': [32, 32, 1],
    'classes': CLASSES,
    'split_groups': SPLIT_GROUPS,
    'files_by_split': {name: [str(p) for g in groups for p in font_groups[g]] for name, groups in SPLIT_GROUPS.items()},
    'font_sha256': font_hashes,
    'sample_counts': {'train': len(y_train), 'validation': len(y_val), 'test_reserved': N_TEST_PER_CLASS * 36},
    'val_data_seed': SEED + 202,
    'validation_arrays_sha256': validation_digest,
    'training_generator': 'V3 + moderate morphology/shear + uniform group sampling',
    'evaluation_generator': 'identique V3 sur les mêmes neuf Lato et quatre DejaVu',
    'input_contract': 'float32 [0,1], 32x32x1, caractère clair sur fond sombre = 1 - gray/255',
    'numpy': np.__version__, 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__,
    'pillow': getattr(Image, '__version__', 'unknown'),
    'test_evaluated': False,
    'limits': 'Synthétique seulement ; version des polices et Pillow peut modifier les images. Test jamais utilisé pour choix de modèle.',
}
(RUN_DIR / 'manifest_v31.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('SHA-256 des arrays de validation :', validation_digest)

## 5 — Vérification visuelle des données TRAIN

Regarder en particulier les formes de **J, I, U, 1, 4** dans différentes polices. Si plusieurs rendus sont illisibles ou coupés, il faut corriger la génération **avant** de lancer l'entraînement. Ne pas afficher des exemples du test final.

In [ ]:
fig, axs = plt.subplots(4, 9, figsize=(14, 6))
for idx, ax in enumerate(axs.flat):
    j = int(np.flatnonzero(y_train == idx)[0])
    ax.imshow(X_train[j, :, :, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(CLASSES[idx], fontsize=10)
    ax.axis('off')
plt.suptitle('CNN V3.1 — échantillons de TRAIN (36 classes)')
plt.tight_layout()
plt.savefig(RUN_DIR / 'samples_train_v31.png', dpi=140, bbox_inches='tight')
plt.show()

focus = 'JIU14'
fig, axs = plt.subplots(len(focus), 8, figsize=(13, 8))
for i, char in enumerate(focus):
    available = np.flatnonzero(y_train == CLASSES.index(char))
    for j, ax in enumerate(axs[i]):
        sample_idx = int(available[j * 17])
        ax.imshow(X_train[sample_idx, :, :, 0], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(char, rotation=0, labelpad=12, fontsize=14)
plt.suptitle('Caractères difficiles — TRAIN seulement')
plt.tight_layout()
plt.savefig(RUN_DIR / 'focus_samples_train_v31.png', dpi=140, bbox_inches='tight')
plt.show()

## 6 — CNN V3.1

**Architecture identique à V3** pour réduire le nombre de variables nouvelles. Aucun `Rescaling(1/255)` : les images sont déjà normalisées, et les augmentations Keras n'agissent qu'à l'entraînement.

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_v31():
    augmentation = models.Sequential([
        layers.RandomRotation(0.025, fill_mode='constant', fill_value=0.0),
        layers.RandomTranslation(0.04, 0.04, fill_mode='constant', fill_value=0.0),
        layers.RandomZoom(0.04, fill_mode='constant', fill_value=0.0),
    ], name='augmentation_train_only')
    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name='character_32x32')
    x = augmentation(inputs)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='class_probabilities')(x)
    model = models.Model(inputs, outputs, name='platevision_cnn_v31')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model_v31 = build_cnn_v31()
model_v31.summary()
assert model_v31.input_shape == (None, 32, 32, 1)
assert model_v31.output_shape == (None, 36)
print('Nombre de paramètres :', model_v31.count_params())

## 7 — Entraîner V3.1

La pondération est **légère**, et concerne uniquement la perte d'entraînement : `J = 1,35`, puis `I`, `U`, `1` et `4` à `1,15`. Les performances de validation restent calculées **sans** poids. L'entraînement est limité à 30 époques avec checkpoint du meilleur `val_loss`, EarlyStopping et réduction du learning rate.

In [ ]:
CLASS_WEIGHTS = {c: 1.0 for c in CLASSES}
CLASS_WEIGHTS.update({'J': 1.35, 'I': 1.15, 'U': 1.15, '1': 1.15, '4': 1.15})
sample_weights = np.asarray([CLASS_WEIGHTS[CLASSES[int(y)]] for y in y_train], dtype=np.float32)
manifest['training_class_weights'] = CLASS_WEIGHTS
(RUN_DIR / 'manifest_v31.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

HISTORY_PATH = RUN_DIR / 'history_v31.json'
if RUN_TRAIN:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(CHECKPOINT), monitor='val_loss', save_best_only=True, mode='min', verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, mode='min', restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
        tf.keras.callbacks.CSVLogger(str(RUN_DIR / 'epochs_v31.csv')),
    ]
    hist_obj = model_v31.fit(
        X_train, y_train,
        sample_weight=sample_weights,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        shuffle=True,
        verbose=1,
    )
    history = {k: [float(x) for x in vals] for k, vals in hist_obj.history.items()}
    HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding='utf-8')
    print('Checkpoint sauvegardé :', CHECKPOINT)
else:
    print('RUN_TRAIN=False ; aucune écriture de checkpoint.')

## 8 — Résultats de validation V3.1

Cette cellule est la **seule évaluation obligatoire** après l'entraînement. Nous regardons accuracy, macro-F1, perte et surtout le rappel de `J`. Une bonne accuracy globale ne suffit pas si `J` demeure presque toujours classé autrement.

In [ ]:
if not CHECKPOINT.exists():
    raise RuntimeError('Modèle V3.1 absent : entraîner le CNN avant de valider.')

best_v31 = tf.keras.models.load_model(str(CHECKPOINT))
val_loss, val_acc = best_v31.evaluate(X_val, y_val, batch_size=256, verbose=0)
pred_v31 = np.argmax(best_v31.predict(X_val, batch_size=256, verbose=0), axis=1)
report_v31 = classification_report(
    y_val, pred_v31, labels=np.arange(36), target_names=list(CLASSES),
    zero_division=0, output_dict=True,
)
cm_v31 = confusion_matrix(y_val, pred_v31, labels=np.arange(36))
metrics_v31 = {
    'validation_loss': float(val_loss),
    'validation_accuracy': float(val_acc),
    'macro_f1': float(report_v31['macro avg']['f1-score']),
    'focus_recall': {c: float(report_v31[c]['recall']) for c in 'JIU14'},
    'classification_report': report_v31,
    'confusion_matrix': cm_v31.tolist(),
    'dataset_sha256': validation_digest,
    'warning': 'Validation synthétique Lato ; ne pas assimiler à une accuracy sur plaques réelles.',
}
(RUN_DIR / 'validation_metrics_v31.json').write_text(json.dumps(metrics_v31, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'V3.1 : accuracy={val_acc:.2%}, val_loss={val_loss:.4f}, macro F1={metrics_v31["macro_f1"]:.2%}')
print('Rappel des caractères difficiles :')
for char in 'JIU14':
    print(f'  {char}: {report_v31[char]["recall"]:.2%} ({int(cm_v31[CLASSES.index(char), CLASSES.index(char)])}/100)')
print('\nConfusions J les plus fréquentes :')
j_row = cm_v31[CLASSES.index('J')]
for idx in np.argsort(j_row)[::-1][:7]:
    if j_row[idx]:
        print(f'  J -> {CLASSES[idx]} : {int(j_row[idx])}')

## 9 — Comparer honnêtement V3 et V3.1 sur **exactement** les mêmes images

Pour comparer, il faut charger **le vrai checkpoint V3** (contenu dans l'archive `20261020T221558Z_8dc39f64_export.zip` ou tout autre ZIP avec `manifest_v3.json` et `platevision_cnn_v3.keras`). Dépose le ZIP dans l'explorateur de fichiers Colab (`/content`) **avant** d'exécuter cette cellule. Ou mets `UPLOAD_V3_ZIP_NOW = True` pour ouvrir une boîte d'import.

Sans le checkpoint, le notebook montre les **chiffres V3 historiques à titre informatif**, sans les présenter comme un calcul sur le même dataset dans cette session.

In [ ]:
UPLOAD_V3_ZIP_NOW = False  # Si nécessaire, activer puis exécuter cette cellule manuellement.

if UPLOAD_V3_ZIP_NOW:
    try:
        from google.colab import files
        uploaded = files.upload()
        print('Fichiers reçus :', list(uploaded))
    except ImportError:
        print('Hors Colab : copier manuellement le ZIP V3 dans le dossier de travail.')

search_paths = [Path('/content'), Path.cwd()]
candidates = []
for base in search_paths:
    if base.exists():
        candidates.extend(sorted(base.glob('*.zip')))

baseline_zip = None
for candidate in dict.fromkeys(candidates):
    try:
        with zipfile.ZipFile(candidate) as z:
            if {'manifest_v3.json', 'platevision_cnn_v3.keras', 'classes.json'} <= set(z.namelist()):
                baseline_zip = candidate
                break
    except (OSError, zipfile.BadZipFile):
        pass

baseline_stats = None
if baseline_zip is None:
    print('Checkpoint V3 non fourni. Charger son ZIP sous /content puis réexécuter cette cellule.')
    print('Référence HISTORIQUE V3 : accuracy 91,86 %, perte 0,3011, rappel J 8 % sur Lato.')
else:
    print('Archive V3 trouvée :', baseline_zip)
    with zipfile.ZipFile(baseline_zip) as z:
        original_manifest = json.loads(z.read('manifest_v3.json'))
        expected_splits = original_manifest.get('split_groups', {})
        assert expected_splits.get('validation') == ['Lato']
        assert expected_splits.get('test') == ['DejaVu']
        assert json.loads(z.read('classes.json')) == list(CLASSES)
        expected_val = sorted(Path(p).name for p in original_manifest['font_files_used']['validation'])
        actual_val = sorted(p.name for p in font_groups['Lato'])
        assert expected_val == actual_val, 'Polices de validation non conformes au manifeste V3.'
        with tempfile.TemporaryDirectory() as tdir:
            baseline_path = Path(tdir) / 'platevision_cnn_v3.keras'
            baseline_path.write_bytes(z.read('platevision_cnn_v3.keras'))
            baseline_model = tf.keras.models.load_model(str(baseline_path))
            base_loss, base_acc = baseline_model.evaluate(X_val, y_val, batch_size=256, verbose=0)
            base_pred = np.argmax(baseline_model.predict(X_val, batch_size=256, verbose=0), axis=1)
            del baseline_model
    base_report = classification_report(
        y_val, base_pred, labels=np.arange(36), target_names=list(CLASSES),
        zero_division=0, output_dict=True,
    )
    baseline_stats = {
        'archive': str(baseline_zip), 'dataset_sha256': validation_digest,
        'v3': {
            'validation_accuracy': float(base_acc), 'validation_loss': float(base_loss),
            'macro_f1': float(base_report['macro avg']['f1-score']),
            'focus_recall': {c: float(base_report[c]['recall']) for c in 'JIU14'},
        },
        'v31': {
            'validation_accuracy': float(val_acc), 'validation_loss': float(val_loss),
            'macro_f1': float(report_v31['macro avg']['f1-score']),
            'focus_recall': {c: float(report_v31[c]['recall']) for c in 'JIU14'},
        },
    }
    (RUN_DIR / 'comparison_v3_v31_same_validation.json').write_text(
        json.dumps(baseline_stats, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    print(f'MÊME VALIDATION : V3 {base_acc:.2%} | V3.1 {val_acc:.2%}')
    print(f'Macro F1 : V3 {base_report["macro avg"]["f1-score"]:.2%} | V3.1 {report_v31["macro avg"]["f1-score"]:.2%}')
    print(f'Rappel J : V3 {base_report["J"]["recall"]:.2%} | V3.1 {report_v31["J"]["recall"]:.2%}')
    print('La validation actuelle a été régénérée ; comparer manifestes, versions et hachages de polices pour confirmer sa fidélité historique.')

## 10 — Courbes, diagnostics et critères de décision

Un candidat V3.1 est intéressant s'il **augmente nettement le rappel de J sans faire s'effondrer les rappels des autres classes**, et s'il reste compétitif sur accuracy et macro-F1. Une métrique favorable isolée ne suffit pas. Une nouvelle modification fondée sur cette même validation augmente le risque de sur-ajustement à Lato.

In [ ]:
if HISTORY_PATH.exists():
    h = json.loads(HISTORY_PATH.read_text(encoding='utf-8'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, ax in [('loss', axes[0]), ('accuracy', axes[1])]:
        ax.plot(h[name], label='Entraînement')
        ax.plot(h['val_' + name], label='Validation Lato')
        ax.set_title(name.capitalize())
        ax.set_xlabel('Époque')
        ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'training_curves_v31.png', dpi=145, bbox_inches='tight')
    plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
focus_chars = 'JIU14'
values = [report_v31[c]['recall'] * 100 for c in focus_chars]
ax.bar(list(focus_chars), values)
ax.set_ylim(0, 100)
ax.set_ylabel('Rappel (%)')
ax.set_title('V3.1 — rappel des caractères difficiles (validation Lato)')
for i, value in enumerate(values):
    ax.text(i, min(97, value + 2), f'{value:.0f}%', ha='center')
plt.tight_layout()
plt.savefig(RUN_DIR / 'focus_recall_v31.png', dpi=145, bbox_inches='tight')
plt.show()

print('Décision recommandée : vérifier les résultats ci-dessus avant de passer au jeu de test DejaVu.')

## 11 — Export V3.1 pour Codex / Streamlit

L'archive inclut le checkpoint V3.1, les classes, le contrat d'entrée, le manifeste, les courbes et les métriques de **validation** (plus la comparaison V3/V3.1 uniquement si le checkpoint V3 a été chargé). Les données synthétiques complètes et les poids de V3 ne sont pas recopiés dans l'archive.

**Important :** télécharger l'archive avant que Colab supprime ses fichiers temporaires.

In [ ]:
(RUN_DIR / 'classes.json').write_text(json.dumps(list(CLASSES), indent=2), encoding='utf-8')
(RUN_DIR / 'INFERENCE_CONTRACT.txt').write_text(
    'PlateVision CNN V3.1 — CLASSIFICATION de caractères isolés (pas de détection de plaque).\n'
    'Entrée: (batch,32,32,1) en float32, plage [0,1].\n'
    'Prétraitement: grayscale uint8, resize 32x32, puis 1.0 - image/255.0; glyphe clair sur fond sombre.\n'
    'Classes: 0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ (indices 0 à 35).\n'
    'Augmentations actives uniquement lors du fit(), pas lors de predict().\n'
    'Scores softmax NON calibrés. Validation synthétique Lato, pas de preuve d’efficacité réelle.\n',
    encoding='utf-8',
)

def export_v31():
    if not CHECKPOINT.exists():
        raise RuntimeError('Checkpoint V3.1 absent, pas d’export possible.')
    archive_path = RUN_DIR.parent / (RUN_DIR.name + '_export.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=5) as z:
        for file in sorted(RUN_DIR.iterdir()):
            if file.is_file():
                z.write(file, arcname=file.name)
    print('Export :', archive_path, '|', round(archive_path.stat().st_size / (2**20), 2), 'Mio')
    return archive_path

ARCHIVE_PATH = export_v31()
if DOWNLOAD_EXPORT:
    try:
        from google.colab import files
        files.download(str(ARCHIVE_PATH))
    except ImportError:
        print('Hors Colab : archive accessible au chemin affiché ci-dessus.')

## 12 — TEST FINAL DejaVu — **BLOQUÉ PAR DÉFAUT**

**NE PAS activer cette cellule tant que tu envisages de modifier le modèle.** Une seule évaluation sur DejaVu doit suivre le gel des choix (architecture, augmentations, hyperparamètres, poids, seuils et politique de sélection). Le test n'est jamais consulté pour choisir V3.1.

Si les modèles sont gelés, passer `RUN_FINAL_TEST` à `True` et exécuter **cette cellule seulement**. Le jeu de test sera généré alors, avec le générateur V3 et `SEED + 303`. La dernière cellule d'export peut ensuite être relancée pour inclure les métriques test dans l'archive (mais ne pas réentraîner).

In [ ]:
RUN_FINAL_TEST = False  # Passer à True UNE SEULE FOIS après avoir figé la sélection.

TEST_METRICS = RUN_DIR / 'test_metrics_v31.json'
if not RUN_FINAL_TEST:
    print('Test final DejaVu verrouillé. Rien n’a été généré ni évalué.')
elif TEST_METRICS.exists():
    print('Test déjà calculé pour cette session :', TEST_METRICS)
elif not CHECKPOINT.exists():
    raise RuntimeError('Checkpoint V3.1 absent.')
else:
    # Génération seulement ici : jamais pendant le développement.
    X_test, y_test, raw_test = generate_split('test', N_TEST_PER_CLASS, SEED + 303, training=False)
    h_test = {hashlib.sha256(a.tobytes()).digest() for a in raw_test}
    for name, known in [('train', h_train), ('validation', h_val)]:
        overlap = len(h_test & known)
        if overlap:
            raise RuntimeError(f'Doublons exacts test/{name} : {overlap}. Protocole invalidé.')
    final_model = tf.keras.models.load_model(str(CHECKPOINT))
    test_loss, test_accuracy = final_model.evaluate(X_test, y_test, batch_size=256, verbose=0)
    test_prediction = np.argmax(final_model.predict(X_test, batch_size=256, verbose=0), axis=1)
    report = classification_report(
        y_test, test_prediction, labels=np.arange(NUM_CLASSES),
        target_names=list(CLASSES), zero_division=0, output_dict=True,
    )
    payload = {
        'test_loss': float(test_loss),
        'test_accuracy': float(test_accuracy),
        'macro_f1': float(report['macro avg']['f1-score']),
        'classification_report': report,
        'confusion_matrix': confusion_matrix(y_test, test_prediction, labels=np.arange(NUM_CLASSES)).tolist(),
        'disclaimer': 'Test synthétique DejaVu. Ce score ne représente pas la lecture de plaques réelles.',
    }
    TEST_METRICS.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print(f'TEST FINAL V3.1 : accuracy={test_accuracy:.2%} | loss={test_loss:.4f}')
    print('Rappel J :', f'{report["J"]["recall"]:.2%}')
    print('Résultats sauvegardés :', TEST_METRICS)

## 13 — Suite du projet

- Comparer V3 et V3.1 **sur les mêmes 3 600 images synthétiques Lato**, via l'import du checkpoint V3. Sans ce checkpoint, ne pas présenter une comparaison chiffrée recalculée.
- Examiner les erreurs des 36 classes, particulièrement `J`, et décider si une nouvelle itération est vraiment utile. Ne pas ajuster les paramètres sur le test final DejaVu.
- Si V3.1 est retenu, transmettre **l'archive ZIP** à Codex pour une intégration **expérimentale** dans Streamlit en conservant V2/V3 et EasyOCR. Vérifier strictement le prétraitement `32×32×1` / `[0,1]`.
- Tester plus tard sur un **jeu indépendant de caractères de plaques réelles correctement annotés**, avec droits d'utilisation, et mesurer séparément la détection, la segmentation et la classification. Une seule image `J389NLT` n'est pas un benchmark.